In [ ]:
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent
OFFICIAL = ROOT / "official" / "more-than-a-hack-2027"
STARTER = OFFICIAL / "starter"

os.environ["MTAH_DATA_DIR"] = str(ROOT / "data" / "raw")

sys.path.insert(0, str(OFFICIAL))
sys.path.insert(0, str(STARTER))

from baseline import (
    build,
    smooth,
    team_of_each_track,
    team_names
)

print("Project:", ROOT)
print("Official repo:", OFFICIAL)

In [ ]:
start_time = time.time()

raw_codes = build()

elapsed = time.time() - start_time

print("Processing completed")
print("Frames:", len(raw_codes))
print("Execution time:", round(elapsed, 2), "seconds")

print("\nRaw predictions:")
print("Home:", int((raw_codes == 0).sum()))
print("Away:", int((raw_codes == 1).sum()))
print("No answer:", int((raw_codes == -1).sum()))

assert len(raw_codes) == 199319
assert set(np.unique(raw_codes)).issubset({-1, 0, 1})

In [ ]:
raw_labels = np.where(
    raw_codes == 0,
    "H",
    np.where(raw_codes == 1, "A", "D")
)

print(pd.Series(raw_labels).value_counts())

assert len(raw_labels) == 199319

In [ ]:
predictions = {}

for window in [25, 50, 125]:

    predictions[window] = smooth(
        raw_codes,
        window
    )

    print(
        f"\nWindow: {window}",
        pd.Series(predictions[window]).value_counts().to_dict()
    )

In [ ]:
def get_possession_statistics(labels):

    labels = np.asarray(labels)

    possessions = []

    current_label = None
    start = None

    for i, label in enumerate(labels):

        if label in ("H", "A"):

            if label != current_label:

                if current_label is not None:
                    possessions.append({
                        "team": current_label,
                        "start": start,
                        "end": i - 1,
                        "duration": i - start
                    })

                current_label = label
                start = i

        else:

            if current_label is not None:
                possessions.append({
                    "team": current_label,
                    "start": start,
                    "end": i - 1,
                    "duration": i - start
                })

            current_label = None
            start = None

    if current_label is not None:
        possessions.append({
            "team": current_label,
            "start": start,
            "end": len(labels) - 1,
            "duration": len(labels) - start
        })

    return pd.DataFrame(possessions)


for window, labels in predictions.items():

    stats = get_possession_statistics(labels)

    print("\nWindow:", window)
    print("Possessions:", len(stats))

    if len(stats):
        print("Mean duration (seconds):",
              round(stats["duration"].mean() / 25, 2))

        print("Median duration (seconds):",
              round(stats["duration"].median() / 25, 2))

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 9))

for ax, window in zip(axes, [25, 50, 125]):

    labels = predictions[window]

    colors = np.where(
        labels == "H", 0,
        np.where(labels == "A", 1, 2)
    )

    ax.plot(colors[:7500], linewidth=0.7)

    ax.set_title(f"Possession prediction - window {window}")
    ax.set_ylabel("H=0, A=1, D=2")
    ax.set_ylim(-0.2, 2.2)

plt.xlabel("Frame")
plt.tight_layout()
plt.show()

In [ ]:
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

np.save(
    PROCESSED / "raw_codes.npy",
    raw_codes
)

for window, labels in predictions.items():

    np.save(
        PROCESSED / f"predictions_smooth_{window}.npy",
        labels
    )

print("Processed files saved successfully")